# Data Exploration: Raw Datasets and Final Analysis Dataset

This notebook explores the six raw datasets used in `GP_level_analysis.ipynb` and the
final analysis dataset produced from them. For each dataset it reports:

- Shape (rows × columns), unit of observation, key columns
- Missing-value counts
- Distributions of key numeric variables

At the end, two summary tables are produced and saved as `.tex` files:

1. **Data availability table** — observations, variables, coverage, time span, unit of obs
2. **Summary statistics table** — mean, SD, min, median, max for key variables in each dataset


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

BASE = '/Users/vidhi/VSCODE/Econ_191_paper/FinalPaperMaterial copy'

PATHS = {
    'nrega':        f'{BASE}/2. Data/1. Raw/Kjelsrud/ReplicationPackage/Datafiles/nrega.dta',
    'gp_id_names':  f'{BASE}/2. Data/1. Raw/Kjelsrud/gp_id_names.dta',
    'mis':          f'{BASE}/2. Data/1. Raw/Mittal/mis_avg_sc_st_data.csv',
    'panchayat_cat':f'{BASE}/2. Data/1. Raw/Mittal/panchayat_category.csv',
    'pc11_shrid':   f'{BASE}/2. Data/1. Raw/SHRUG/shrug-pc-keys-csv/pc11r_shrid_key.csv',
    'shrid_names':  f'{BASE}/2. Data/1. Raw/SHRUG/shrug-shrid-keys-csv/shrid_loc_names.csv',
    'final':        f'{BASE}/2. Data/3. Processed/gp_analysis_dataset.csv',
}

OUTPUTS = f'{BASE}/4. Outputs'

# Shared helper: compact shape + missing summary
def shape_summary(df, name):
    print(f"{'─'*60}")
    print(f"Dataset: {name}")
    print(f"  Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Duplicate rows : {df.duplicated().sum():,}")
    miss = df.isnull().sum()
    miss = miss[miss > 0]
    if len(miss):
        print(f"  Columns with missing values ({len(miss)}):")
        for col, n in miss.items():
            print(f"    {col:<35} {n:>8,}  ({100*n/len(df):.1f}%)")
    else:
        print("  No missing values")


## 1. `nrega.dta` — Kjelsrud Replication Package

**Unit of observation:** Gram Panchayat × year (2011–2017)  
**Source:** Kjelsrud (2024) replication package  
**Role in analysis:** Provides the treatment variable (`fragmentation_2004_past`),
fixed-effects cell (`pc_dist`), delimitation flag (`change_pc`), and all nine
Census 2001 control variables. Collapsed to GP level (one row per GP) before merging.


In [2]:
nrega = pd.read_stata(PATHS['nrega'])
shape_summary(nrega, 'nrega.dta')
print()
print("Column list:")
print(list(nrega.columns))


────────────────────────────────────────────────────────────
Dataset: nrega.dta
  Shape          : 451,231 rows × 55 columns


  Duplicate rows : 0
  Columns with missing values (1):
    distance_km                           62,808  (13.9%)

Column list:
['state', 'district2011', 'gp_id', 'year', 'pc_id_pre', 'pc_id_post', 'change_pc', 'matched_pc_post', 'postbank', 'sPostbank', 'demand', 'sDemand', 'gini_pre66_past', 'meanlog_pre66_past', 'theil_pre66_past', 'palma_pre66_past', 'fragmentation_2004_past', 'min_vote_share_2004_past', 'min_margin_2004_past', 'effective_parties_2004_past', 'reservation_post', 'party_2009', 'log_mean_pre66_past', 'pc_dist', 'cell_share_sc', 'cell_share_st', 'cell_share_l6', 'cell_share_lit', 'cell_primary', 'cell_phc', 'cell_phs', 'cell_tap', 'cell_power', 'cell_paved', 'primary_past', 'phc_past', 'phs_past', 'tap_past', 'power_past', 'paved_past', 'share_sc_past', 'share_st_past', 'share_l6_past', 'population_past', 'urbanization_past', 'share_lit_past', 'religion_frac_past', 'caste_frac_past', 'share_blocksplit', 'distance_km', 'change_ac', 'split_block', 'poverty_pre66_past', '

In [3]:
# ── Key numeric variables ──────────────────────────────────────────────────
key_nrega = ['fragmentation_2004_past', 'change_pc', 'share_l6_past',
             'share_lit_past', 'poverty_pre66_past', 'ln_population',
             'urbanization_past', 'primary_past', 'phc_past', 'paved_past', 'power_past']
key_nrega = [c for c in key_nrega if c in nrega.columns]
print("Key variable summary (all GP-years):")
display(nrega[key_nrega].describe().T.round(3))


Key variable summary (all GP-years):


,count,mean,std,min,25%,50%,75%,max
fragmentation_2004_past,451231.000,0.660,0.085,0.438,0.587,0.658,0.733,0.810
change_pc,451231.000,0.269,0.444,0.000,0.000,0.000,1.000,1.000
share_l6_past,451231.000,0.168,0.030,0.101,0.142,0.169,0.196,0.227
share_lit_past,451231.000,0.529,0.103,0.239,0.461,0.540,0.602,0.857
poverty_pre66_past,451231.000,0.337,0.178,0.003,0.187,0.339,0.471,0.825
urbanization_past,451231.000,0.214,0.137,0.000,0.114,0.193,0.269,0.983
primary_past,451231.000,0.854,0.132,0.419,0.788,0.895,0.952,1.000
phc_past,451231.000,0.044,0.046,0.000,0.020,0.030,0.055,0.800
paved_past,451231.000,0.643,0.241,0.192,0.443,0.656,0.872,1.000
power_past,451231.000,0.578,0.330,0.000,0.265,0.581,0.919,1.000


In [4]:
# ── Panel structure ────────────────────────────────────────────────────────
print("Years present:", sorted(nrega['year'].unique()))
print(f"GPs (unique gp_id): {nrega['gp_id'].nunique():,}")
print("States (unique):", nrega['state'].nunique())
print(f"GP-year obs: {len(nrega):,}")

# Year distribution
fig, ax = plt.subplots(figsize=(7, 3))
nrega['year'].value_counts().sort_index().plot(kind='bar', ax=ax,
    color='steelblue', edgecolor='white')
ax.set_title('nrega.dta — Observations per Year', fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('GP-year rows')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(f'{OUTPUTS}/eda_nrega_years.png', dpi=150, bbox_inches='tight')
plt.show()
print("States covered:", sorted(nrega['state'].astype(int).unique()))


Years present: [np.float32(2011.0), np.float32(2012.0), np.float32(2013.0)]
GPs (unique gp_id): 150,413
States (unique): 14
GP-year obs: 451,231
States covered: [np.int64(3), np.int64(6), np.int64(9), np.int64(10), np.int64(19), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(27), np.int64(28), np.int64(29), np.int64(32), np.int64(33)]


## 2. `gp_id_names.dta` — GP Name Strings

**Unit of observation:** Gram Panchayat (one row per GP)  
**Source:** Kjelsrud replication package  
**Role in analysis:** Provides the raw GP, state, district, and block name strings
used for the three-key name match to Mittal data.


In [5]:
gpnames = pd.read_stata(PATHS['gp_id_names'])
shape_summary(gpnames, 'gp_id_names.dta')
print()
display(gpnames.head(8))


────────────────────────────────────────────────────────────
Dataset: gp_id_names.dta
  Shape          : 151,660 rows × 5 columns


  Duplicate rows : 0
  No missing values



,gpname,state2011,district2011,cdblock2011,gp_id
0,nadimikella,28,542,444,128294.000
1,kambara,28,542,444,128289.000
2,dasumantha puram,28,542,444,128286.000
3,narsipuram,28,542,444,128296.000
4,chalivendri,28,542,444,128283.000
5,nadukuru,28,542,444,128295.000
6,vikrampuram,28,542,444,128305.000
7,chittipudivalasa,28,542,444,128285.000


In [6]:
print(f"Unique gp_id values : {gpnames['gp_id'].nunique():,}")
print("Duplicate gp_id     :", gpnames['gp_id'].duplicated().sum())
print("Unique states       :", gpnames['state2011'].nunique())
print("Unique districts    :", gpnames['district2011'].nunique())

# Sample length distributions of GP names
fig, ax = plt.subplots(figsize=(7, 3))
gpnames['gpname'].str.len().dropna().hist(bins=40, ax=ax, color='teal', edgecolor='white')
ax.set_title('gp_id_names.dta — Distribution of GP Name String Lengths', fontsize=10)
ax.set_xlabel('Characters in GP name'); ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(f'{OUTPUTS}/eda_gpnames_lengths.png', dpi=150, bbox_inches='tight')
plt.show()


Unique gp_id values : 151,660
Duplicate gp_id     : 0
Unique states       : 14
Unique districts    : 412


## 3. `mis_avg_sc_st_data.csv` — Mittal SC/ST Person-Days

**Unit of observation:** Gram Panchayat (identified by state / district / panchayat name)  
**Source:** Mittal et al. — MIS-MGNREGA SC/ST employment data  
**Role in analysis:** Provides the **numerator** of the targeting ratio —
average SC and ST person-days worked per GP.  
After name-matching, ~119K of 151K GPs are matched (~79%).


In [7]:
mis = pd.read_csv(PATHS['mis'])
shape_summary(mis, 'mis_avg_sc_st_data.csv')
print()
print("Columns:", list(mis.columns))
display(mis.head(5))


────────────────────────────────────────────────────────────
Dataset: mis_avg_sc_st_data.csv
  Shape          : 295,262 rows × 13 columns
  Duplicate rows : 0
  No missing values

Columns: ['state', 'district', 'block', 'panchayat', 'JC_issued_SC', 'JC_issued_ST', 'JC_issued_oth', 'emp_provided_household_sc', 'emp_provided_household_st', 'emp_provided_household_oth', 'emp_provided_persondays_sc', 'emp_provided_persondays_st', 'emp_provided_persondays_oth']


,state,district,block,panchayat,JC_issued_SC,JC_issued_ST,JC_issued_oth,emp_provided_household_sc,emp_provided_household_st,emp_provided_household_oth,emp_provided_persondays_sc,emp_provided_persondays_st,emp_provided_persondays_oth
0,andaman and nicobar,nicobars,campbell bay,campbell bay,4.000,2.000,385.714,0.571,0.143,2.571,4.714,2.286,69.429
1,andaman and nicobar,nicobars,campbell bay,govindnagar,0.000,0.000,435.571,0.000,0.000,8.857,0.000,0.000,106.429
2,andaman and nicobar,nicobars,campbell bay,great & little nicobar,0.000,27.857,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,andaman and nicobar,nicobars,campbell bay,laxmi nagar,0.000,0.000,472.286,0.000,0.000,4.143,0.000,0.000,30.571
4,andaman and nicobar,nicobars,nancowry,chowra tc,0.000,309.571,0.000,0.000,0.000,0.000,0.000,0.000,0.000


In [8]:
# ── Key person-day variables ────────────────────────────────────────────────
pd_cols = [c for c in mis.columns if 'emp_provided' in c or 'JC_issued' in c or 'personday' in c.lower()]
if pd_cols:
    display(mis[pd_cols].describe().T.round(1))
    
print(f"\nUnique states    : {mis['state'].nunique()}")
print(f"Unique districts : {mis['district'].nunique()}")
print(f"Unique panchayats: {mis['panchayat'].nunique():,}")

# Distribution of total SC person-days (log scale)
sc_col = [c for c in mis.columns if 'sc' in c.lower() and 'emp' in c.lower()]
st_col = [c for c in mis.columns if 'st' in c.lower() and 'emp' in c.lower() and 'sc' not in c.lower()]

if sc_col and st_col:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for ax, col, color, label in zip(axes,
                                     [sc_col[0], st_col[0]],
                                     ['#2176AE', '#8B3A8C'],
                                     ['SC person-days', 'ST person-days']):
        vals = mis[col].replace(0, np.nan).dropna()
        np.log1p(vals).hist(bins=50, ax=ax, color=color, edgecolor='white', alpha=0.85)
        ax.set_title(f'log(1+{label})', fontsize=10)
        ax.set_xlabel('log(1 + person-days)'); ax.set_ylabel('GPs')
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.suptitle('mis_avg_sc_st_data.csv — Person-Day Distributions', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS}/eda_mis_persondays.png', dpi=150, bbox_inches='tight')
    plt.show()


,count,mean,std,min,25%,50%,75%,max
JC_issued_SC,295262.000,95.900,185.400,0.000,7.100,41.100,113.300,6700.400
JC_issued_ST,295262.000,61.200,176.300,0.000,0.000,1.400,39.000,22004.000
JC_issued_oth,295262.000,299.800,477.100,0.000,56.600,157.900,332.300,11047.600
emp_provided_household_sc,295262.000,39.100,93.100,0.000,1.400,13.900,42.300,3503.600
emp_provided_household_st,295262.000,30.100,93.600,0.000,0.000,0.300,15.000,8803.400
emp_provided_household_oth,295262.000,112.000,199.800,0.000,13.100,50.000,126.900,6385.600
emp_provided_persondays_sc,295262.000,1828.600,4840.600,0.000,47.600,530.900,1773.200,214462.300
emp_provided_persondays_st,295262.000,1538.400,4988.000,0.000,0.000,9.400,649.600,313716.900
emp_provided_persondays_oth,295262.000,5300.500,10447.500,0.000,447.900,1997.300,5789.700,490223.600



Unique states    : 34
Unique districts : 721
Unique panchayats: 212,012


## 4. `panchayat_category.csv` — Mittal SC/ST Population Counts

**Unit of observation:** Gram Panchayat (identified by state / district / subdistrict / panchayat name)  
**Source:** Mittal et al. — Census-based SC/ST population data at panchayat level  
**Role in analysis:** Provides the **denominator** of the targeting ratio —
SC and ST population counts per GP. This is the **binding constraint** on sample size:
only ~65K of 151K GPs are matched (~43%).


In [9]:
pancat = pd.read_csv(PATHS['panchayat_cat'])
shape_summary(pancat, 'panchayat_category.csv')
print()
print("Columns:", list(pancat.columns))
display(pancat.head(5))


────────────────────────────────────────────────────────────
Dataset: panchayat_category.csv
  Shape          : 127,696 rows × 12 columns
  Duplicate rows : 0
  Columns with missing values (2):
    District                                 134  (0.1%)
    Panchayat                                  1  (0.0%)

Columns: ['State', 'District', 'Subdistrict', 'Panchayat', 'MIS Index', 'Number of Households', 'SC population', 'ST population', 'General population', 'Total population', 'Panchayat Type', 'Panchayat Category']


,State,District,Subdistrict,Panchayat,MIS Index,Number of Households,SC population,ST population,General population,Total population,Panchayat Type,Panchayat Category
0,maharashtra,satara,Satara,kalambe,186822,367,122,0,1567,1689,3,7
1,maharashtra,satara,Satara,kameri,186824,383,87,45,1637,1769,3,7
2,maharashtra,satara,Satara,kanher,186826,414,68,0,1887,1955,3,7
3,maharashtra,satara,Satara,karandi,186827,258,126,22,991,1139,3,7
4,maharashtra,parbhani,Palam,pokharnidevi,189471,329,410,0,1230,1640,1,3


In [10]:
pop_cols = [c for c in pancat.columns if any(k in c for k in ['SC','ST','population','Population','household','Household'])]
if pop_cols:
    display(pancat[pop_cols].describe().T.round(1))

print(f"\nUnique states        : {pancat['State'].nunique()}")
print(f"Unique districts     : {pancat['District'].nunique()}")
print(f"Unique panchayats    : {pancat['Panchayat'].nunique():,}")

# SC and ST population shares
sc_pop = pancat['SC population'] if 'SC population' in pancat.columns else None
st_pop = pancat['ST population'] if 'ST population' in pancat.columns else None
tot_col = [c for c in pancat.columns if 'total' in c.lower() or 'population' in c.lower() and 'SC' not in c and 'ST' not in c]

if sc_pop is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    sc_pop.replace(0, np.nan).dropna().clip(upper=sc_pop.quantile(0.99)).hist(
        bins=50, ax=axes[0], color='#2176AE', edgecolor='white', alpha=0.85)
    axes[0].set_title('SC Population per GP (clipped at 99th pct)', fontsize=10)
    axes[0].set_xlabel('SC population'); axes[0].set_ylabel('GPs')
    
    if st_pop is not None:
        st_pop.replace(0, np.nan).dropna().clip(upper=st_pop.quantile(0.99)).hist(
            bins=50, ax=axes[1], color='#8B3A8C', edgecolor='white', alpha=0.85)
        axes[1].set_title('ST Population per GP (clipped at 99th pct)', fontsize=10)
        axes[1].set_xlabel('ST population'); axes[1].set_ylabel('GPs')
    
    for ax in axes:
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.suptitle('panchayat_category.csv — Population Distributions', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS}/eda_pancat_populations.png', dpi=150, bbox_inches='tight')
    plt.show()


,count,mean,std,min,25%,50%,75%,max
Number of Households,127696.000,705.100,874.100,1.000,264.000,423.000,800.000,38588.000
SC population,127696.000,610.000,1152.000,0.000,82.000,296.000,690.000,31264.000
ST population,127696.000,425.400,948.200,0.000,0.000,33.000,402.000,38600.000
General population,127696.000,2339.500,3251.000,0.000,690.000,1355.000,2629.000,275051.000
Total population,127696.000,3374.800,4142.000,1.000,1275.000,2026.000,3775.000,314000.000



Unique states        : 31
Unique districts     : 577
Unique panchayats    : 100,360


## 5. SHRUG Crosswalk Files

Two files are used together to map numeric Census 2011 state/district codes (in `nrega.dta`)
to string names (needed for the three-key name match to Mittal data).

### 5a. `pc11r_shrid_key.csv`
Maps SHRUG rural location identifiers (`shrid2`) to Census 2011 numeric codes
(state, district, sub-district, village).

### 5b. `shrid_loc_names.csv`
Maps the same `shrid2` to human-readable state, district, and sub-district name strings.

**Together:** `pc11r_shrid_key` joined on `shrid2` → deduplicate to unique
`(pc11_state_id, pc11_district_id)` pairs → lookup table of numeric code → name string.


In [11]:
shrid_key   = pd.read_csv(PATHS['pc11_shrid'])
shrid_names = pd.read_csv(PATHS['shrid_names'])

shape_summary(shrid_key,   'pc11r_shrid_key.csv')
print()
shape_summary(shrid_names, 'shrid_loc_names.csv')


────────────────────────────────────────────────────────────
Dataset: pc11r_shrid_key.csv
  Shape          : 597,597 rows × 7 columns


  Duplicate rows : 0
  Columns with missing values (2):
    pc11_subdistrict_id                        1  (0.0%)
    pc11_land_area                             9  (0.0%)

────────────────────────────────────────────────────────────
Dataset: shrid_loc_names.csv
  Shape          : 596,389 rows × 7 columns


  Duplicate rows : 0
  Columns with missing values (4):
    subdistrict_name                         150  (0.0%)
    town_name                            588,720  (98.7%)
    village_name                           7,670  (1.3%)
    place_name                                 1  (0.0%)


In [12]:
print("pc11r_shrid_key.csv columns  :", list(shrid_key.columns))
print("shrid_loc_names.csv columns  :", list(shrid_names.columns))
print()
print(f"Unique shrid2 in key file  : {shrid_key['shrid2'].nunique():,}")
print(f"Unique shrid2 in names file: {shrid_names['shrid2'].nunique():,}")
print(f"Overlap (inner join)       : {len(pd.merge(shrid_key[['shrid2']].drop_duplicates(), shrid_names[['shrid2']].drop_duplicates(), on='shrid2')):,}")

# Resulting crosswalk after deduplication
crosswalk = (
    pd.merge(shrid_key[['shrid2','pc11_state_id','pc11_district_id']],
             shrid_names[['shrid2','state_name','district_name']],
             on='shrid2', how='inner')
    .drop_duplicates(subset=['pc11_state_id','pc11_district_id'])
)
print(f"\nFinal crosswalk unique (state_id, district_id) pairs: {len(crosswalk):,}")
print("Coverage: states =", crosswalk['pc11_state_id'].nunique(),
      " districts =", crosswalk['pc11_district_id'].nunique())
display(crosswalk.head(6))


pc11r_shrid_key.csv columns  : ['shrid2', 'pc11_state_id', 'pc11_district_id', 'pc11_subdistrict_id', 'pc11_village_id', 'pc11_land_area', 'pc11_pca_tot_p']
shrid_loc_names.csv columns  : ['shrid2', 'state_name', 'district_name', 'subdistrict_name', 'town_name', 'village_name', 'place_name']

Unique shrid2 in key file  : 588,978


Unique shrid2 in names file: 596,389


Overlap (inner join)       : 588,974



Final crosswalk unique (state_id, district_id) pairs: 631
Coverage: states = 35  districts = 631


,shrid2,pc11_state_id,pc11_district_id,state_name,district_name
0,11-01-001-00001-000001,1,1,jammu kashmir,kupwara
353,11-01-002-00004-000371,1,2,jammu kashmir,badgam
815,11-01-003-00010-000848,1,3,jammu kashmir,leh ladakh
926,11-01-004-00013-000962,1,4,jammu kashmir,kargil
1051,11-01-005-00016-001089,1,5,jammu kashmir,punch
1221,11-01-006-00020-001268,1,6,jammu kashmir,rajouri


## 6. `gp_analysis_dataset.csv` — Final Analysis Dataset

**Unit of observation:** Gram Panchayat (one row per GP)  
**Generated by:** Section 15 of `GP_level_analysis.ipynb`  
**Role:** Standalone dataset from which all regressions in the paper can be reproduced.  
Contains treatment variable, FE cell, controls, targeting ratio outcomes,
population shares, sample flags, and pre-computed variables.


In [13]:
final = pd.read_csv(PATHS['final'])
shape_summary(final, 'gp_analysis_dataset.csv')
print()
print("Columns:", list(final.columns))


────────────────────────────────────────────────────────────
Dataset: gp_analysis_dataset.csv
  Shape          : 47,995 rows × 33 columns
  Duplicate rows : 0
  Columns with missing values (4):
    ln_sc_targeting                        3,080  (6.4%)
    ln_st_targeting                       19,990  (41.7%)
    sc_targeting_ratio                     1,499  (3.1%)
    st_targeting_ratio                    14,544  (30.3%)

Columns: ['gp_id', 'state', 'state_name', 'district2011', 'pc_id_pre', 'pc_id_post', 'pc_dist', 'change_pc', 'fragmentation_2004_past', 'fragmentation_std', 'fragmentation_std_sq', 'ln_sc_targeting', 'ln_st_targeting', 'ln_scst_targeting', 'sc_targeting_ratio', 'st_targeting_ratio', 'scst_targeting_ratio', 'sc_pop_share', 'st_pop_share', 'in_st_sample', 'has_sc_outcome', 'has_st_outcome', 'share_l6_past', 'share_lit_past', 'poverty_pre66_past', 'ln_population', 'urbanization_past', 'primary_past', 'phc_past', 'paved_past', 'power_past', 'share_sc_past', 'share_st_past'

In [14]:
# ── Outcome variable distributions ─────────────────────────────────────────
outcome_cols = ['ln_sc_targeting', 'ln_st_targeting', 'ln_scst_targeting']
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['#2176AE', '#8B3A8C', '#1A7A4A']
titles = ['Log SC Targeting\n(winsorised)', 'Log ST Targeting\n(winsorised)', 'Log SC+ST Targeting\n(winsorised)']

for ax, col, color, title in zip(axes, outcome_cols, colors, titles):
    vals = final[col].dropna()
    vals.hist(bins=60, ax=ax, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', lw=1.2, ls='--', label='Ratio = 1')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('log targeting ratio'); ax.set_ylabel('GPs')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend(fontsize=8)

plt.suptitle('Final Dataset — Outcome Variable Distributions', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUTS}/eda_final_outcomes.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"SC outcome non-missing  : {final['ln_sc_targeting'].notna().sum():,} ({100*final['ln_sc_targeting'].notna().mean():.1f}%)")
print(f"ST outcome non-missing  : {final['ln_st_targeting'].notna().sum():,} ({100*final['ln_st_targeting'].notna().mean():.1f}%)")
print(f"SC+ST outcome non-missing: {final['ln_scst_targeting'].notna().sum():,} ({100*final['ln_scst_targeting'].notna().mean():.1f}%)")
print(f"ST sub-sample (pop≥5%)  : {final['in_st_sample'].sum():,} ({100*final['in_st_sample'].mean():.1f}%)")


SC outcome non-missing  : 44,915 (93.6%)
ST outcome non-missing  : 28,005 (58.3%)
SC+ST outcome non-missing: 47,995 (100.0%)
ST sub-sample (pop≥5%)  : 18,563 (38.7%)


In [15]:
# ── Treatment variable distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

final['fragmentation_2004_past'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Political Competition (raw EC)', fontsize=10)
axes[0].set_xlabel('fragmentation_2004_past (1 − HHI)'); axes[0].set_ylabel('GPs')

final['fragmentation_std'].hist(bins=50, ax=axes[1], color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', lw=1.2, ls='--', label='Mean = 0')
axes[1].set_title('Political Competition (standardised)', fontsize=10)
axes[1].set_xlabel('fragmentation_std (z-score)'); axes[1].set_ylabel('GPs')
axes[1].legend(fontsize=8)

for ax in axes:
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Final Dataset — Treatment Variable', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUTS}/eda_final_treatment.png', dpi=150, bbox_inches='tight')
plt.show()

print("Reshuffled by delimitation (change_pc=1):", final['change_pc'].sum(), f"({100*final['change_pc'].mean():.1f}%)")
print("State distribution:")
display(final.groupby('state_name').size().rename('GP count').reset_index().sort_values('GP count', ascending=False))


Reshuffled by delimitation (change_pc=1): 11709.0 (24.4%)
State distribution:


,state_name,GP count
7,Madhya Pradesh,9334
8,Maharashtra,8548
3,Gujarat,5752
0,Andhra Pradesh,4116
12,Uttar Pradesh,3988
2,Chhattisgarh,3023
11,Tamil Nadu,2925
1,Bihar,2170
9,Odisha,1982
10,Punjab,1749


In [16]:
# ── Controls distributions ──────────────────────────────────────────────────
controls = ['share_l6_past','share_lit_past','poverty_pre66_past',
            'ln_population','urbanization_past','primary_past',
            'phc_past','paved_past','power_past']
ctrl_labels = ['Under-6 share','Literacy rate','Poverty headcount',
               'Log pop.','Urbanisation','Primary school',
               'PHC available','Paved road','Electricity']

fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col, lbl in zip(axes.flat, controls, ctrl_labels):
    final[col].hist(bins=40, ax=ax, color='#4A7FA5', edgecolor='white', alpha=0.85)
    ax.set_title(lbl, fontsize=9)
    ax.set_xlabel(col, fontsize=7.5)
    ax.set_ylabel('GPs', fontsize=8)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Final Dataset — Control Variable Distributions', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUTS}/eda_final_controls.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Summary Table 1: Data Availability

One row per dataset. Columns: dataset name, source, unit of observation,
total rows, columns, key identifier, geographic coverage, time coverage, role in analysis.


In [17]:
avail_rows = [
    {
        'Dataset': 'nrega.dta',
        'Source': 'Kjelsrud (2024)',
        'Unit of Obs.': 'GP × year',
        'Rows': f'{len(nrega):,}',
        'Columns': nrega.shape[1],
        'Unique GPs': f'{nrega["gp_id"].nunique():,}',
        'States': nrega['state'].nunique(),
        'Years': f'{int(nrega["year"].min())}–{int(nrega["year"].max())}',
        'Role': 'Treatment, FE cell, controls',
    },
    {
        'Dataset': 'gp_id_names.dta',
        'Source': 'Kjelsrud (2024)',
        'Unit of Obs.': 'GP',
        'Rows': f'{len(gpnames):,}',
        'Columns': gpnames.shape[1],
        'Unique GPs': f'{gpnames["gp_id"].nunique():,}',
        'States': gpnames['state2011'].nunique(),
        'Years': 'Cross-section',
        'Role': 'GP name strings for matching',
    },
    {
        'Dataset': 'mis_avg_sc_st_data.csv',
        'Source': 'Mittal et al.',
        'Unit of Obs.': 'GP (by name)',
        'Rows': f'{len(mis):,}',
        'Columns': mis.shape[1],
        'Unique GPs': f'{mis["panchayat"].nunique():,}',
        'States': mis['state'].nunique(),
        'Years': 'Avg. 2011–2013',
        'Role': 'SC/ST person-days (numerator)',
    },
    {
        'Dataset': 'panchayat_category.csv',
        'Source': 'Mittal et al.',
        'Unit of Obs.': 'GP (by name)',
        'Rows': f'{len(pancat):,}',
        'Columns': pancat.shape[1],
        'Unique GPs': f'{pancat["Panchayat"].nunique():,}',
        'States': pancat['State'].nunique(),
        'Years': 'Cross-section',
        'Role': 'SC/ST population (denominator)',
    },
    {
        'Dataset': 'pc11r_shrid_key.csv',
        'Source': 'SHRUG',
        'Unit of Obs.': 'SHRUG location (shrid2)',
        'Rows': f'{len(shrid_key):,}',
        'Columns': shrid_key.shape[1],
        'Unique GPs': f'{shrid_key["shrid2"].nunique():,}',
        'States': shrid_key['pc11_state_id'].nunique(),
        'Years': 'Cross-section',
        'Role': 'Numeric code → shrid crosswalk',
    },
    {
        'Dataset': 'shrid_loc_names.csv',
        'Source': 'SHRUG',
        'Unit of Obs.': 'SHRUG location (shrid2)',
        'Rows': f'{len(shrid_names):,}',
        'Columns': shrid_names.shape[1],
        'Unique GPs': f'{shrid_names["shrid2"].nunique():,}',
        'States': shrid_names['state_name'].nunique(),
        'Years': 'Cross-section',
        'Role': 'shrid → string names crosswalk',
    },
    {
        'Dataset': 'gp_analysis_dataset.csv',
        'Source': 'This paper (Section 15)',
        'Unit of Obs.': 'GP',
        'Rows': f'{len(final):,}',
        'Columns': final.shape[1],
        'Unique GPs': f'{final["gp_id"].nunique():,}',
        'States': final['state_name'].nunique(),
        'Years': 'Cross-section',
        'Role': 'Final analysis dataset',
    },
]

avail_df = pd.DataFrame(avail_rows)
print("Table 1: Data Availability")
display(avail_df)


Table 1: Data Availability


,Dataset,Source,Unit of Obs.,Rows,Columns,Unique GPs,States,Years,Role
0,nrega.dta,Kjelsrud (2024),GP × year,"451,231",55,"150,413",14,2011–2013,"Treatment, FE cell, controls"
1,gp_id_names.dta,Kjelsrud (2024),GP,"151,660",5,"151,660",14,Cross-section,GP name strings for matching
2,mis_avg_sc_st_data.csv,Mittal et al.,GP (by name),"295,262",13,"212,012",34,Avg. 2011–2013,SC/ST person-days (numerator)
3,panchayat_category.csv,Mittal et al.,GP (by name),"127,696",12,"100,360",31,Cross-section,SC/ST population (denominator)
4,pc11r_shrid_key.csv,SHRUG,SHRUG location (shrid2),"597,597",7,"588,978",35,Cross-section,Numeric code → shrid crosswalk
5,shrid_loc_names.csv,SHRUG,SHRUG location (shrid2),"596,389",7,"596,389",35,Cross-section,shrid → string names crosswalk
6,gp_analysis_dataset.csv,This paper (Section 15),GP,"47,995",33,"47,995",14,Cross-section,Final analysis dataset


In [18]:
# ── Save Table 1 as LaTeX ──────────────────────────────────────────────────
def tex_row(*cols):
    return ' & '.join(str(c) for c in cols) + ' \\\\'

col_names = list(avail_df.columns)

lines = [
    '\\begin{table}[htbp]',
    '\\centering',
    '\\caption{Data Availability: Raw Datasets and Final Analysis Dataset}',
    '\\label{tab:data_availability}',
    '\\small',
    '\\begin{tabular}{llllrrrll}',
    '\\hline\\hline',
    tex_row(*['\\textbf{'+c+'}' for c in col_names]),
    '\\hline',
]

for _, row in avail_df.iterrows():
    vals = [str(v).replace('_', '\\_') for v in row.values]
    lines.append(tex_row(*vals))

lines += [
    '\\hline\\hline',
    '\\multicolumn{9}{p{18cm}}{\\footnotesize \\textit{Notes:} '
    'GP = Gram Panchayat. SHRUG = Socioeconomic High-resolution Rural-Urban Geographic dataset. '
    'Mittal et al. person-day data covers 2011--2013 MGNREGA records. '
    'The final analysis dataset is the result of merging all raw sources and applying sample restrictions '
    '(non-missing treatment, FE cell, nine controls, at least one outcome variable).}',
    '\\end{tabular}',
    '\\end{table}',
]

tex = '\n'.join(lines)
out = f'{OUTPUTS}/table_data_availability.tex'
with open(out, 'w') as f:
    f.write(tex)
print(f'Saved: {out}')
print()
print(tex)


Saved: /Users/vidhi/VSCODE/Econ_191_paper/FinalPaperMaterial copy/4. Outputs/table_data_availability.tex

\begin{table}[htbp]
\centering
\caption{Data Availability: Raw Datasets and Final Analysis Dataset}
\label{tab:data_availability}
\small
\begin{tabular}{llllrrrll}
\hline\hline
\textbf{Dataset} & \textbf{Source} & \textbf{Unit of Obs.} & \textbf{Rows} & \textbf{Columns} & \textbf{Unique GPs} & \textbf{States} & \textbf{Years} & \textbf{Role} \\
\hline
nrega.dta & Kjelsrud (2024) & GP × year & 451,231 & 55 & 150,413 & 14 & 2011–2013 & Treatment, FE cell, controls \\
gp\_id\_names.dta & Kjelsrud (2024) & GP & 151,660 & 5 & 151,660 & 14 & Cross-section & GP name strings for matching \\
mis\_avg\_sc\_st\_data.csv & Mittal et al. & GP (by name) & 295,262 & 13 & 212,012 & 34 & Avg. 2011–2013 & SC/ST person-days (numerator) \\
panchayat\_category.csv & Mittal et al. & GP (by name) & 127,696 & 12 & 100,360 & 31 & Cross-section & SC/ST population (denominator) \\
pc11r\_shrid\_key.csv & SHR

## 8. Summary Table 2: Summary Statistics by Dataset

Key numeric variables from each dataset, with N, mean, SD, min, median, max.
Grouped by dataset for easy comparison.


In [19]:
def var_stats(df, col, label, dataset):
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    return {
        'Dataset':  dataset,
        'Variable': label,
        'N':        f'{len(s):,}',
        'Mean':     f'{s.mean():.3f}',
        'SD':       f'{s.std():.3f}',
        'Min':      f'{s.min():.3f}',
        'Median':   f'{s.median():.3f}',
        'Max':      f'{s.max():.3f}',
    }

rows_stats = []

# nrega.dta
for col, lbl in [
    ('fragmentation_2004_past', 'Political competition (EC)'),
    ('change_pc',               'Reshuffled by delimitation'),
    ('share_l6_past',           'Under-6 population share'),
    ('share_lit_past',          'Literacy rate'),
    ('poverty_pre66_past',      'Poverty headcount'),
]:
    if col in nrega.columns:
        rows_stats.append(var_stats(nrega.drop_duplicates('gp_id'), col, lbl, 'nrega.dta'))

# mis_avg_sc_st_data.csv
sc_emp_col = next((c for c in mis.columns if 'emp' in c.lower() and 'sc' in c.lower()), None)
st_emp_col = next((c for c in mis.columns if 'emp' in c.lower() and '_st' in c.lower()), None)
if sc_emp_col:
    rows_stats.append(var_stats(mis, sc_emp_col, f'SC person-days ({sc_emp_col})', 'mis_avg_sc_st_data.csv'))
if st_emp_col:
    rows_stats.append(var_stats(mis, st_emp_col, f'ST person-days ({st_emp_col})', 'mis_avg_sc_st_data.csv'))

# panchayat_category.csv
for col, lbl in [('SC population', 'SC population'), ('ST population', 'ST population')]:
    if col in pancat.columns:
        rows_stats.append(var_stats(pancat, col, lbl, 'panchayat_category.csv'))

# final dataset
for col, lbl in [
    ('fragmentation_std',    'Political competition (std.)'),
    ('ln_sc_targeting',      'Log SC targeting ratio'),
    ('ln_st_targeting',      'Log ST targeting ratio'),
    ('ln_scst_targeting',    'Log SC+ST targeting ratio'),
    ('sc_targeting_ratio',   'SC targeting ratio (raw)'),
    ('st_targeting_ratio',   'ST targeting ratio (raw)'),
    ('scst_targeting_ratio', 'SC+ST targeting ratio (raw)'),
    ('share_l6_past',        'Under-6 population share'),
    ('share_lit_past',       'Literacy rate'),
    ('poverty_pre66_past',   'Poverty headcount'),
    ('ln_population',        'Log rural population'),
    ('urbanization_past',    'Urbanisation rate'),
]:
    if col in final.columns:
        rows_stats.append(var_stats(final, col, lbl, 'gp_analysis_dataset.csv'))

stats_df = pd.DataFrame(rows_stats)
print("Table 2: Summary Statistics by Dataset")
display(stats_df)


Table 2: Summary Statistics by Dataset


,Dataset,Variable,N,Mean,SD,Min,Median,Max
0,nrega.dta,Political competition (EC),"150,413",0.660,0.085,0.438,0.658,0.810
1,nrega.dta,Reshuffled by delimitation,"150,413",0.269,0.444,0.000,0.000,1.000
2,nrega.dta,Under-6 population share,"150,413",0.168,0.030,0.101,0.169,0.227
3,nrega.dta,Literacy rate,"150,413",0.529,0.103,0.239,0.540,0.857
4,nrega.dta,Poverty headcount,"150,413",0.337,0.178,0.003,0.339,0.825
5,mis_avg_sc_st_data.csv,SC person-days (emp_provided_household_sc),"295,262",39.114,93.080,0.000,13.857,3503.571
6,mis_avg_sc_st_data.csv,ST person-days (emp_provided_household_st),"295,262",30.101,93.554,0.000,0.286,8803.429
7,panchayat_category.csv,SC population,"127,696",609.962,1152.032,0.000,296.000,31264.000
8,panchayat_category.csv,ST population,"127,696",425.376,948.232,0.000,33.000,38600.000
9,gp_analysis_dataset.csv,Political competition (std.),"47,995",-0.213,0.932,-2.612,-0.191,1.757


In [20]:
# ── Save Table 2 as LaTeX ──────────────────────────────────────────────────
col_names2 = list(stats_df.columns)

lines2 = [
    '\\begin{table}[htbp]',
    '\\centering',
    '\\caption{Summary Statistics by Dataset}',
    '\\label{tab:summary_stats_datasets}',
    '\\small',
    '\\begin{tabular}{llrrrrrr}',
    '\\hline\\hline',
    tex_row(*['\\textbf{'+c+'}' for c in col_names2]),
    '\\hline',
]

prev_dataset = None
for _, row in stats_df.iterrows():
    if row['Dataset'] != prev_dataset:
        if prev_dataset is not None:
            lines2.append('\\hline')
        lines2.append('\\multicolumn{8}{l}{\\textit{' + str(row['Dataset']).replace('_','\\_') + '}} \\\\')
        prev_dataset = row['Dataset']
    vals = ['', str(row['Variable']).replace('_', '\\_')] + [str(row[c]) for c in col_names2[2:]]
    lines2.append(tex_row(*vals))

lines2 += [
    '\\hline\\hline',
    '\\multicolumn{8}{p{14cm}}{\\footnotesize \\textit{Notes:} '
    'For nrega.dta, statistics are computed at the GP level (de-duplicated on gp\\_id). '
    'For mis\\_avg\\_sc\\_st\\_data.csv and panchayat\\_category.csv, all rows are used. '
    'For gp\\_analysis\\_dataset.csv, statistics are over the 47{,}995-GP base sample; '
    'outcome variables have fewer non-missing observations due to name-matching constraints.}',
    '\\end{tabular}',
    '\\end{table}',
]

tex2 = '\n'.join(lines2)
out2 = f'{OUTPUTS}/table_summary_stats_datasets.tex'
with open(out2, 'w') as f:
    f.write(tex2)
print(f'Saved: {out2}')
print()
print(tex2)


Saved: /Users/vidhi/VSCODE/Econ_191_paper/FinalPaperMaterial copy/4. Outputs/table_summary_stats_datasets.tex

\begin{table}[htbp]
\centering
\caption{Summary Statistics by Dataset}
\label{tab:summary_stats_datasets}
\small
\begin{tabular}{llrrrrrr}
\hline\hline
\textbf{Dataset} & \textbf{Variable} & \textbf{N} & \textbf{Mean} & \textbf{SD} & \textbf{Min} & \textbf{Median} & \textbf{Max} \\
\hline
\multicolumn{8}{l}{\textit{nrega.dta}} \\
 & Political competition (EC) & 150,413 & 0.660 & 0.085 & 0.438 & 0.658 & 0.810 \\
 & Reshuffled by delimitation & 150,413 & 0.269 & 0.444 & 0.000 & 0.000 & 1.000 \\
 & Under-6 population share & 150,413 & 0.168 & 0.030 & 0.101 & 0.169 & 0.227 \\
 & Literacy rate & 150,413 & 0.529 & 0.103 & 0.239 & 0.540 & 0.857 \\
 & Poverty headcount & 150,413 & 0.337 & 0.178 & 0.003 & 0.339 & 0.825 \\
\hline
\multicolumn{8}{l}{\textit{mis\_avg\_sc\_st\_data.csv}} \\
 & SC person-days (emp\_provided\_household\_sc) & 295,262 & 39.114 & 93.080 & 0.000 & 13.857 & 3503

## 9. Sample Attrition: From Raw Data to Final Sample

Shows how the sample size shrinks at each step of the data pipeline.


In [21]:
steps = [
    ('nrega.dta\n(all GP-years)',          nrega['gp_id'].count()),
    ('nrega.dta\n(unique GPs)',            nrega['gp_id'].nunique()),
    ('After: non-missing\ntreatment + FE + controls',
      final[['fragmentation_std','pc_dist'] + ['share_l6_past','share_lit_past',
       'poverty_pre66_past','ln_population','urbanization_past',
       'primary_past','phc_past','paved_past','power_past']].dropna().shape[0]),
    ('Final base sample\n(≥1 outcome)',    len(final)),
    ('SC regression\n(non-missing SC)',    final['ln_sc_targeting'].notna().sum()),
    ('SC+ST regression\n(non-missing)',    final['ln_scst_targeting'].notna().sum()),
    ('ST regression\n(ST pop ≥5%)',        int(final['in_st_sample'].sum())),
]

labels = [s[0] for s in steps]
counts = [s[1] for s in steps]

fig, ax = plt.subplots(figsize=(12, 4.5))
colors = ['#2176AE','#2176AE','#F4A742','#1A7A4A','#1A7A4A','#1A7A4A','#8B3A8C']
bars = ax.barh(range(len(labels)), counts, color=colors, edgecolor='white', height=0.6)

for i, (bar, n) in enumerate(zip(bars, counts)):
    ax.text(n + max(counts)*0.01, i, f'{n:,}', va='center', fontsize=9.5, color='#333333')

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Number of observations (GPs)', fontsize=10)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title('Sample Attrition from Raw Data to Analysis Sample', fontsize=11, fontweight='bold')
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, max(counts) * 1.18)

plt.tight_layout()
plt.savefig(f'{OUTPUTS}/eda_sample_attrition.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sample attrition summary:")
for lbl, n in steps:
    print(f"  {lbl.replace(chr(10),' '):<50} {n:>10,}")


Sample attrition summary:
  nrega.dta (all GP-years)                              451,231
  nrega.dta (unique GPs)                                150,413
  After: non-missing treatment + FE + controls           47,995
  Final base sample (≥1 outcome)                         47,995
  SC regression (non-missing SC)                         44,915
  SC+ST regression (non-missing)                         47,995
  ST regression (ST pop ≥5%)                             18,563
